# Week 07 Friday: Synthesis Day
Author: Student

### 1. Label Distribution & Evaluation Metrics

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import time
import os

try:
    df = pd.read_csv('"C:\\Users\\avish\\Desktop\\iitgn AINPT\\week-7\\shopsense_reviews.csv"')
    print("Dataset Loaded.")
except Exception as e:
    # Fallback to local if running in the folder
    df = pd.read_csv('product_reviews.csv') if os.path.exists('product_reviews.csv') else None

if df is not None:
    dist = df['sentiment_label'].value_counts(normalize=True) * 100
    print("Class Distribution (%):\n", dist)
    
# Explanation: Why simple accuracy is unreliable?
# Accuracy treats all classes and errors equally. If a dataset is heavily skewed (e.g. 90% positive),
# a naive model predicting 'positive' all the time achieves 90% accuracy but is completely useless
# at identifying negative reviews, which are typically the most critical for action.


### 2. Evaluate Sentiment Classifier

In [ ]:
# We will use Vader as a quick baseline "trained" classifier
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def predict_vader_label(text):
    score = analyzer.polarity_scores(str(text))['compound']
    if score >= 0.05: return 'positive'
    elif score <= -0.05: return 'negative'
    else: return 'neutral'

# Sample 2000 reviews for speed
sample_df = df.sample(2000, random_state=42)
sample_df['pred_vader'] = sample_df['review_text'].apply(predict_vader_label)

print("--- VADER PERFORMANCE ---")
print(classification_report(sample_df['sentiment_label'], sample_df['pred_vader']))

# Stakeholder Summary:
# "Our baseline system successfully identifies positive reviews but struggles slightly with neutral and negative nuances. 
# While overall accuracy is stable, we need to focus on 'Recall' for Negative reviews, meaning the percentage of 
# actual complaints that our system successfully catches, as missing them costs customer trust."


### 3. Compare Against Constraints (Categories, Code-Mixing, Latency)

In [ ]:
# Constraint evaluation:
# 1. Generalization to new categories: TF-IDF + LR degrades if vocabulary shifts. VADER relies on static rules so might be more robust if language doesn't change.
# 2. Hindi-English code-mixing: Neither plain VADER nor basic English TF-IDF handles this natively.
# 3. Inference latency:
start = time.time()
_ = [predict_vader_label(t) for t in sample_df['review_text']]
vader_time = (time.time() - start) / len(sample_df) * 1000  # in ms
print(f"VADER Latency: {vader_time:.2f} ms per review")

# Conclusion: VADER meets the <20ms latency constraint constraint easily. However, it fails on Hindi-English code-mixing.
# A distilled multilingual transformer (like mDistilBERT) would handle code-mixing and category generalization,
# but we would need ONNX/TensorRT optimization to meet the <20ms latency constraint.


### 4. Cost Model Analysis

In [ ]:
# Defined Cost Model:
# - False Negative (Negative flagged as Positive): $50 cost (Missed complaint -> churned customer)
# - False Positive (Positive flagged as Negative): $2 cost (Unnecessary manual review by CS agent)

def calculate_daily_cost(y_true, y_pred, volume=100000):
    cm = confusion_matrix(y_true, y_pred, labels=['negative', 'positive', 'neutral'])
    # cm[0, 1] means actual negative, predicted positive -> False Negative
    # cm[1, 0] means actual positive, predicted negative -> False Positive
    
    fn_rate = cm[0, 1] / len(y_true)
    fp_rate = cm[1, 0] / len(y_true)
    
    daily_fn_cost = (fn_rate * volume) * 50
    daily_fp_cost = (fp_rate * volume) * 2
    
    return daily_fn_cost, daily_fp_cost

if 'pred_vader' in sample_df.columns:
    subset = sample_df[sample_df['sentiment_label'].isin(['negative', 'positive'])]
    fn_cost, fp_cost = calculate_daily_cost(subset['sentiment_label'], subset['pred_vader'])
    print(f"Projected Daily FN Cost: ${fn_cost:.2f}")
    print(f"Projected Daily FP Cost: ${fp_cost:.2f}")
    print(f"Total Daily Misclassification Cost: ${fn_cost + fp_cost:.2f}")


### 5. Technical Brief & Recommendation
**Recommendation:** We do not roll out the standard VADER or basic TF-IDF system. Instead, we deploy a lightweight multilingual model (distilled), applying structural threshold tuning to heavily penalize False Negatives.

**Tracking:** 
- **Metric:** Weekly F1-Score for the 'Negative' class, plus real-time drift detection on prediction confidence distribution.
- **Threshold for Retraining:** If 'Negative Recall' drops below 80% or out-of-vocabulary rate shifts by >15% (new categories/lingo).


### 6 & 7. Hard: Root Cause of "Predict Positive for Everything" 

In [ ]:
# Simulated failure model that predicts majority class (positive)
sample_df['broken_pred'] = 'positive'

broken_fn, broken_fp = calculate_daily_cost(subset['sentiment_label'], subset['broken_pred'])
print(f"BROKEN PREDICTION COST")
print(f"Projected Daily FN Cost (Broken): ${broken_fn:.2f}")
print(f"Projected Daily FP Cost (Broken): ${broken_fp:.2f}")
print(f"Total Daily Misclassification Cost: ${broken_fn + broken_fp:.2f}")

# Explanation of Root Cause:
# The deployed classifier achieved 94% accuracy simply because the training data was extremely imbalanced (e.g. 94% positive).
# It learned to minimize cross-entropy loss by predicting the majority class.
# The previous team failed to analyze Confusion Matrices and recall on minority (negative) classes.

# Fix: Use class weighting in loss functions (e.g. class_weight='balanced' in sklearn), SMOTE for oversampling,
# or under-sampling the majority class.
